# DT2-SC · diagnóstico de tinturaria (Google Colab)
Calibra o gêmeo digital com o CSV da sua planta e compara cenários S0–S4.

**Privacidade:** o arquivo enviado fica na sessão do Colab; para dados sensíveis, rode localmente (`python -m dt2sc planta ...`).

In [ ]:
# 1. Instalação: envie o ZIP do projeto (Arquivos > Upload) e descompacte
!unzip -oq DT2-SC_*.zip -d dt2sc_proj
%cd dt2sc_proj/dt2sc
!pip -q install numpy pandas scipy matplotlib pyyaml

In [ ]:
# 2. Dados: envie o CSV da planta (ou gere um exemplo sintético)
from google.colab import files
from dt2sc import planta as P
USAR_EXEMPLO = True   # mude para False para enviar o seu CSV
if USAR_EXEMPLO:
    P.exemplo_csv('planta.csv', 14)
    caminho = 'planta.csv'
else:
    caminho = next(iter(files.upload()))

In [ ]:
# 3. Carregar, mapear colunas e checar qualidade
import pandas as pd
bruto = pd.read_csv(caminho, sep=None, engine='python')
print(P.detectar_mapa(bruto.columns))
MAPA = None          # ex.: {'producao_kg_h': 'kg_hora', 'eletrica_kw': 'P_total'}
FATOR = None         # ex.: {'eletrica_kw': 0.001} se a coluna estiver em W
df = P.carregar(bruto, MAPA, FATOR)
P.qualidade(df)

In [ ]:
# 4. Intensidades, calibração e anomalias
cal = P.calibrar(df)
print(cal['avisos'])
pd.Series(cal['kpis']).to_frame('valor')

In [ ]:
import matplotlib.pyplot as plt
dia = P.intensidades_diarias(df)
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, c in zip(ax, ['eletr_kwh_t', 'gn_m3_t', 'agua_m3_t']):
    dia[c].plot(ax=a, marker='o'); a.set_title(c)
plt.tight_layout()
P.anomalias(df).head(20)

In [ ]:
# 5. Cenários e economia com parâmetros locais
res = P.avaliar_planta(cal, n=3000, tarifa_gas=None, preco_cavaco_r_t=None, gate_fee=None)
print(P.relatorio_md(cal, res, P.anomalias(df), P.qualidade(df)))

In [ ]:
# 6. Baixar relatório
open('relatorio.md', 'w').write(P.relatorio_md(cal, res))
files.download('relatorio.md')